# Reasoning Loops — DebounceHook and Hard Limits

Based on:
- [Language models can overthink](https://the-decoder.com/language-models-can-overthink-and-get-stuck-in-endless-thought-loops/) — The Decoder, Jan 2025
- [How many reasoning steps do AI agents need](https://particula.tech/blog/ai-agent-loops-reasoning-steps-optimization) — Particula, Jul 2025
- [How to Prevent Infinite Loops](https://codieshub.com/for-ai/prevent-agent-loops-costs) — CodiesHub, Dec 2025

## The Problem

Agents loop when tools give ambiguous feedback. If a tool always says "prices may change — try again", the agent has no signal to stop. It retries the same call with the same input, burning tokens with each iteration, until it hits the model's iteration limit.

This is not a bug in the tool. It's a design choice that accidentally removes the stopping condition.

## The Tools

Four tools in `tools.py` — two that **cause** loops, two that **prevent** them:

| Tool | Returns | Effect on agent |
|------|---------|-----------------|
| `search_flights(origin, destination, max_price)` | `"More results may be available. Prices change frequently."` | ❌ Ambiguous — agent retries hoping for better prices |
| `check_hotel_price(hotel, check_in)` | `"Prices may change — check again for latest."` | ❌ Ambiguous — agent retries to get current price |
| `book_flight(flight, passenger)` | `"SUCCESS: FL12345"` or `"FAILED: No seats"` | ✅ Clear — agent stops on SUCCESS |
| `book_hotel(hotel, guest, nights)` | `"SUCCESS: HT67890 confirmed"` or `"FAILED: Fully booked"` | ✅ Clear — agent stops on SUCCESS |

## The Hooks

Two hooks in `hooks.py` intercept tool calls via `BeforeToolCallEvent`:

**`DebounceHook(window_size=3)`** — detects duplicate calls in a sliding window. If the same `(tool_name, input)` pair appears ≥ 2 times in the last `window_size` calls, it cancels the call via `event.cancel_tool`. The agent receives `BLOCKED: Duplicate call` and must try a different approach.

**`LimitToolCounts(max_tool_counts)`** — hard ceiling per tool per invocation. When a tool exceeds its limit, subsequent calls are cancelled. The agent cannot exceed it regardless of LLM behavior.

![Ambiguous Tool Feedback vs DebounceHook + Clear States](../images/Ambiguous-Tool-Feedback.jpg)

## What We Test

| Test | Tools | Hook | Query |
|------|-------|------|-------|
| 1 — Ambiguous (problem) | `search_flights`, `check_hotel_price` | None | Same budget query |
| 2 — DebounceHook (solution) | Same tools | `DebounceHook` | Same budget query |
| 3 — Clear states | `book_flight`, `book_hotel` | None | Booking query |
| 4 — Hard limits | `search_flights`, `check_hotel_price` | `LimitToolCounts(max=2)` | Multi-city query |

Tests 1 and 2 use **the exact same query and tools** — the only variable is the presence of `DebounceHook`.

## 📦 Setup

In [ ]:
import os
import time
os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel
from tools import search_flights, check_hotel_price, book_flight, book_hotel
from hooks import DebounceHook, LimitToolCounts

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys and add OPENAI_API_KEY=your-key to a .env file.")

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# Prompt that causes the agent to retry when it can't find prices within budget —
# this is what triggers organic loops in Scenario 1.
PERSISTENT_PROMPT = (
    "You are a persistent travel agent. Always try to find prices within the user's budget. "
    "If results are over budget, search again — prices fluctuate and you might find better deals on retry."
)

# Scenarios 1 and 2 use the exact same query. Only difference: hooks.
BUDGET_QUERY = "Find me the cheapest flight from NYC to Paris under $400 and a hotel under $200/night for March 15"

def count_tool_calls(agent):
    count = 0
    for msg in agent.messages:
        for block in msg.get("content", []):
            if "toolUse" in block:
                count += 1
    return count

print("✅ Setup complete!")

---

## 🔬 Scenario 1: Baseline (No Loop Detection)

**Research:** "Agent calls same tool repeatedly without progress"

**Expected:** May make redundant calls without detection

In [ ]:
agent_loop = Agent(
    model=MODEL,
    system_prompt=PERSISTENT_PROMPT,
    tools=[search_flights, check_hotel_price],
)

start = time.time()
response = agent_loop(BUDGET_QUERY)
time_loop = time.time() - start
calls_loop = count_tool_calls(agent_loop)

print(f"⏱️  {time_loop:.1f}s — {calls_loop} tool calls")
if calls_loop > 4:
    print(f"⚠️  {calls_loop} calls — ambiguous feedback caused retries")
else:
    print("ℹ️  Agent stopped early (LLM behavior varies run-to-run)")

---

## 🚫 Scenario 2: Debounce Hook (Solution from Research)

**Research Solution:** "Cache/Debounce Layer with Hooks"

**Expected:** Duplicate calls detected and blocked

![How DebounceHook works — flow diagram](../images/How-DebounceHook-Works.jpg)

In [ ]:
debounce = DebounceHook(window_size=3)

agent_debounce = Agent(
    model=MODEL,
    system_prompt=PERSISTENT_PROMPT,
    tools=[search_flights, check_hotel_price],
    hooks=[debounce],  # only change from Scenario 1
)

start = time.time()
response = agent_debounce(BUDGET_QUERY)
time_debounce = time.time() - start

stats = debounce.get_stats()
calls_debounce = stats['total_calls']

print(f"⏱️  {time_debounce:.1f}s — {stats['total_calls']} allowed, {stats['blocked_calls']} blocked")
if stats['blocked_calls'] > 0:
    print(f"✅ DebounceHook blocked {stats['blocked_calls']} duplicate calls")
else:
    print("✅ No duplicates this run (LLM behavior varies)")

---

## ✅ Scenario 3: Clear Success States

**Research:** "Tools return SUCCESS/FAILED, agent knows when to stop"

**Expected:** Agent stops after receiving SUCCESS

In [ ]:
agent_clear = Agent(
    model=MODEL,
    tools=[book_flight, book_hotel],
)

query_book = "Book a flight NYC to Paris for Alex Rivera, and a hotel called Le Marais for 3 nights"

start = time.time()
response = agent_clear(query_book)
time_clear = time.time() - start
calls_clear = count_tool_calls(agent_clear)

print(f"⏱️  {time_clear:.1f}s — {calls_clear} tool calls")
print("✅ SUCCESS states — agent stopped immediately")

---

## 🔢 Scenario 4: Hard Limits

**Research:** "Treat every agent run as bounded process with explicit limits"

**Expected:** Agent stops at reasonable iteration count

In [ ]:
limit_hook = LimitToolCounts(max_tool_counts={
    "search_flights": 2,
    "check_hotel_price": 2,
})

agent_limits = Agent(
    model=MODEL,
    system_prompt="You are a travel agent. Find the best deal for the user.",
    tools=[search_flights, check_hotel_price],
    hooks=[limit_hook],
)

query_multi = "Compare flights and hotels for NYC to Paris, London, and Tokyo — find the cheapest option for each"

start = time.time()
response = agent_limits(query_multi)
time_limits = time.time() - start
calls_limits = sum(limit_hook.tool_counts.values())

print(f"⏱️  {time_limits:.1f}s — tool counts: {limit_hook.tool_counts}")
print("✅ Hard ceiling enforced")

---
## Summary

**Strands Agents makes loop prevention simple**: attach a `HookProvider` to `Agent(hooks=[...])` and Strands intercepts every tool call via `BeforeToolCallEvent` — no external libraries, no agent modification, no custom loop detection code.

```python
# All it takes:
agent = Agent(tools=[search_flights], hooks=[DebounceHook(window_size=3)])
```

### When to Use Each Solution

| Solution | Best for |
|----------|----------|
| **Clear SUCCESS/FAILED states** | Tools you control — design unambiguous terminal states |
| **DebounceHook** | External tools that return partial/changing results |
| **LimitToolCounts** | Hard cost ceilings — non-negotiable regardless of LLM behavior |

### Next Steps

1. ➡️ [Demo 01: Context Overflow](../01-context-overflow-demo/) — Memory Pointer Pattern
2. ➡️ [Demo 02: MCP Timeout](../02-mcp-timeout-demo/) — Async handleId pattern

### References

- [Language models can overthink](https://the-decoder.com/language-models-can-overthink-and-get-stuck-in-endless-thought-loops/) — The Decoder, Jan 2025
- [How many reasoning steps do AI agents need](https://particula.tech/blog/ai-agent-loops-reasoning-steps-optimization) — Particula, Jul 2025

### Strands Agents

- [Strands Hooks](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/hooks/) — `BeforeToolCallEvent`, `cancel_tool`, `HookProvider`
- [Strands Hooks Cookbook](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/hooks/) — `LimitToolCounts` and other patterns
- [Strands Model Providers](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/model-providers/) — Swap to Bedrock, Anthropic, Ollama
- [Strands Agents Documentation](https://strandsagents.com) — Full framework docs
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)